In [2]:
# 1. Install library yang dibutuhkan
!pip install -q transformers torch tqdm pandas

In [3]:
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification, pipeline
from tqdm.auto import tqdm

# 2. Load dataset mentah
FILE_PATH = "dataset_karhutla_raw.csv"
TEXT_COLUMN = "Komentar Asli"

df = pd.read_csv(FILE_PATH)
print(f"Total baris awal: {len(df)}")

# Hapus baris kosong atau duplikat jika ada
df = df.dropna(subset=[TEXT_COLUMN]).drop_duplicates(subset=[TEXT_COLUMN]).reset_index(drop=True)
print(f"Total baris setelah bersih dari null & duplikat: {len(df)}")

Total baris awal: 22701
Total baris setelah bersih dari null & duplikat: 22408


In [4]:
# 3. Load model IndoRoBERTa Sentiment
MODEL_NAME = "w11wo/indonesian-roberta-base-sentiment-classifier"
device = 0 if torch.cuda.is_available() else -1

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME)

sentiment_pipeline = pipeline(
    "sentiment-analysis",
    model=model,
    tokenizer=tokenizer,
    device=device,
    truncation=True,
    max_length=256  # Membatasi panjang token agar cepat & tidak error memori
)

# 4. Jalankan prediksi secara batch (jauh lebih cepat di GPU)
texts = df[TEXT_COLUMN].astype(str).tolist()
batch_size = 64
results = []

for i in tqdm(range(0, len(texts), batch_size), desc="Melabeli komentar"):
    batch = texts[i:i + batch_size]
    preds = sentiment_pipeline(batch)
    results.extend(preds)

# 5. Masukkan hasil label dan confidence score ke DataFrame
df["label_indobert"] = [res["label"] for res in results]
df["confidence"] = [round(res["score"], 4) for res in results]

# Simpan hasil lengkap
df.to_csv("dataset_karhutla_indobert.csv", index=False)
print("Selesai! File 'dataset_karhutla_indobert.csv' berhasil disimpan.")

config.json:   0%|          | 0.00/929 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/328 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/808k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/467k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.38M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

... (2 outputs left out)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Melabeli komentar:   0%|          | 0/351 [00:00<?, ?it/s]

[transformers] You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


Selesai! File 'dataset_karhutla_indobert.csv' berhasil disimpan.


In [5]:
# Cek distribusi label baru
print("=== Distribusi Label IndoBERT ===")
print(df["label_indobert"].value_counts())
print("\n=== Persentase ===")
print((df["label_indobert"].value_counts(normalize=True) * 100).round(2))

# Ambil 250 sampel acak yang high-confidence (> 0.75) untuk kamu cek manual
sample_val = df[df["confidence"] > 0.75].sample(n=250, random_state=42)
sample_val[[TEXT_COLUMN, "label_indobert", "confidence"]].to_csv("sampel_validasi_manual.csv", index=False)
print("\nFile 'sampel_validasi_manual.csv' siap diunduh untuk pengecekan manual.")

=== Distribusi Label IndoBERT ===
label_indobert
negative    17201
positive     3082
neutral      2125
Name: count, dtype: int64

=== Persentase ===
label_indobert
negative    76.76
positive    13.75
neutral      9.48
Name: proportion, dtype: float64

File 'sampel_validasi_manual.csv' siap diunduh untuk pengecekan manual.


#Mengunduh dataset IndoNLU SmSA (Indonesian Sentiment Analysis)

In [6]:
import pandas as pd

# URL langsung ke file dataset IndoNLU SmSA (Train & Valid)
url_train = "https://raw.githubusercontent.com/IndoNLP/indonlu/master/dataset/smsa_doc-sentiment-prosa/train_preprocess.tsv"
url_valid = "https://raw.githubusercontent.com/IndoNLP/indonlu/master/dataset/smsa_doc-sentiment-prosa/valid_preprocess.tsv"

# Baca file TSV (pemisah tab, tanpa header)
df_train = pd.read_csv(url_train, sep="\t", names=["text", "label"])
df_valid = pd.read_csv(url_valid, sep="\t", names=["text", "label"])

# Gabungkan keduanya
df_smsa = pd.concat([df_train, df_valid], ignore_index=True)

print(f"Total data IndoNLU SmSA: {len(df_smsa)} baris")
print(df_smsa["label"].value_counts())

# Simpan ke CSV
df_smsa.to_csv("dataset_general_smsa.csv", index=False)

Total data IndoNLU SmSA: 12260 baris
label
positive    7151
negative    3830
neutral     1279
Name: count, dtype: int64


#Penggabungan dataset scraping(karhutla) dengan dataset IndoNLU

In [8]:
import pandas as pd
from sklearn.model_selection import train_test_split

import os
os.makedirs("data", exist_ok=True)

# 1. Load kedua dataset (Sesuaikan path & nama kolom jika berbeda)
df_karhutla = pd.read_csv("dataset_karhutla_indobert.csv")
df_smsa = pd.read_csv("dataset_general_smsa.csv")

# 2. Standarisasi nama kolom menjadi: ['text', 'label', 'domain']
df_k = df_karhutla[["Komentar Asli", "label_indobert"]].copy()
df_k.columns = ["text", "label"]
df_k["domain"] = "karhutla"

df_s = df_smsa[["text", "label"]].copy()
df_s["domain"] = "general"

# 3. Gabungkan dan bersihkan baris kosong / duplikat
df_all = pd.concat([df_k, df_s], ignore_index=True)
df_all["label"] = df_all["label"].str.lower().str.strip()
df_all = df_all.dropna(subset=["text", "label"]).drop_duplicates(subset=["text"]).reset_index(drop=True)

print("=== Total Distribusi Kelas Gabungan ===")
print(df_all.groupby(["domain", "label"]).size().unstack(fill_value=0))

# 4. Pisahkan Out-of-Domain Test Set (500 sampel acak dari IndoNLU untuk uji lintas topik)
df_general = df_all[df_all["domain"] == "general"]
df_out_test, df_general_rem = train_test_split(
    df_general, train_size=500, stratify=df_general["label"], random_state=42
)

# Gabungkan sisa data general dengan seluruh data karhutla
df_main = pd.concat([df_all[df_all["domain"] == "karhutla"], df_general_rem], ignore_index=True)

# 5. Bagi df_main menjadi Train (80%), Validation (10%), dan In-Domain Test (10%)
df_train, df_temp = train_test_split(
    df_main, test_size=0.20, stratify=df_main["label"], random_state=42
)
df_val, df_in_test = train_test_split(
    df_temp, test_size=0.50, stratify=df_temp["label"], random_state=42
)

# 6. Simpan hasil split ke folder data/
df_train.to_csv("data/train.csv", index=False)
df_val.to_csv("data/val.csv", index=False)
df_in_test.to_csv("data/test_indomain.csv", index=False)
df_out_test.to_csv("data/test_outdomain.csv", index=False)

print(f"\nData berhasil dibagi:")
print(f"- Train          : {len(df_train)} baris")
print(f"- Validation     : {len(df_val)} baris")
print(f"- Test In-Domain : {len(df_in_test)} baris")
print(f"- Test Out-Domain: {len(df_out_test)} baris")

=== Total Distribusi Kelas Gabungan ===
label     negative  neutral  positive
domain                               
general       3798     1267      7108
karhutla     17201     2125      3082

Data berhasil dibagi:
- Train          : 27264 baris
- Validation     : 3408 baris
- Test In-Domain : 3409 baris
- Test Out-Domain: 500 baris


#Modul Preprocessing


In [10]:
import os
import re
import pandas as pd

# URL dataset kamus slang standar (Colloquial Indonesian Lexicon)
SLANG_URL = "https://raw.githubusercontent.com/nasalsabila/kamus-alay/master/colloquial-indonesian-lexicon.csv"
LOCAL_SLANG_PATH = "data/colloquial-indonesian-lexicon.csv"

def load_slang_dict() -> dict:
    # Kamus cadangan dasar jika tidak ada koneksi internet
    slang_dict = {
        "yg": "yang", "dg": "dengan", "dgn": "dengan", "gk": "tidak", "ga": "tidak",
        "gak": "tidak", "nggak": "tidak", "tdk": "tidak", "bgt": "banget",
        "krn": "karena", "utk": "untuk", "dlm": "dalam", "pd": "pada",
        "sm": "sama", "tp": "tapi", "klo": "kalau", "kalo": "kalau",
        "sy": "saya", "gw": "saya", "gue": "saya", "lu": "kamu", "lo": "kamu",
        "udh": "sudah", "udah": "sudah", "blm": "belum", "sdh": "sudah",
        "lg": "lagi", "bkn": "bukan", "jgn": "jangan", "dr": "dari",
        "bs": "bisa", "jd": "jadi", "emg": "memang", "bener": "benar"
    }

    try:
        if os.path.exists(LOCAL_SLANG_PATH):
            df_slang = pd.read_csv(LOCAL_SLANG_PATH)
        else:
            df_slang = pd.read_csv(SLANG_URL)
            os.makedirs("data", exist_ok=True)
            df_slang.to_csv(LOCAL_SLANG_PATH, index=False)

        # Gabungkan 4.300+ kata dari kamus eksternal ke dalam dictionary
        external_dict = dict(zip(df_slang["slang"], df_slang["formal"]))
        slang_dict.update(external_dict)
    except Exception as e:
        print(f"Peringatan: Menggunakan kamus slang dasar karena gagal memuat kamus eksternal ({e})")

    return slang_dict

# Muat kamus sekali saja di memori saat modul di-import
SLANG_DICT = load_slang_dict()

def clean_text(text: str) -> str:
    if not isinstance(text, str):
        return ""

    # 1. Case folding (huruf kecil)
    text = text.lower()

    # 2. Hapus URL, mention (@user), dan hashtag (#topik)
    text = re.sub(r"http\S+|www\S+|https\S+", "", text)
    text = re.sub(r"@\w+|#\w+", "", text)

    # 3. Hapus angka dan tanda baca (sisakan huruf dan spasi)
    text = re.sub(r"[^a-z\s]", " ", text)

    # 4. Normalisasi huruf berulang berlebihan (contoh: "kerennnnn" -> "kerenn")
    text = re.sub(r"(.)\1{2,}", r"\1\1", text)

    # 5. Normalisasi kata tidak baku menggunakan kamus
    words = text.split()
    normalized_words = [SLANG_DICT.get(word, word) for word in words]

    # 6. Rapikan spasi berlebih
    return " ".join(normalized_words).strip()